# Check-in 1 smoke experiment

Tree-Ring and Gaussian Shading on Stable Diffusion 2.1-base: clean baseline, real-photo baseline, and imprint forgery with a same-model attacker.

Use a GPU runtime. Results are written to `OUT`; every stage resumes from `runs.csv` if the session is interrupted, as long as `OUT` still exists. To survive a runtime reset, mount Google Drive and point `OUT` at a Drive folder.

In [ ]:
BRANCH = "checkin-1"
OUT = "results/checkin1"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
%cd /content
if not os.path.isdir("cmpe261-watermark-forgery"):
    !git clone --branch {BRANCH} https://github.com/yashashav-dk/cmpe261-watermark-forgery.git
%cd /content/cmpe261-watermark-forgery
!git pull --ff-only
!pip install -q -e ".[attack]" "markdiffusion @ git+https://github.com/THU-BPM/MarkDiffusion@9d81656d1a5f9e5194fc2f727bb795ef29e53809"
# Colab's preinstalled torchaudio is built for a newer torch than MarkDiffusion's pin allows; nothing here uses it.
!pip uninstall -y -q torchaudio
!python -c "import torch, diffusers, markdiffusion; print('torch', torch.__version__, '| diffusers', diffusers.__version__, '| cuda', torch.cuda.is_available())"

## Dry run

Two prompts, two photographs, one cover, ten imprint steps. Checks the whole path before spending compute.

In [ ]:
!python -m wmforge.smoke --out results/dryrun --n-clean 2 --n-real 2 --n-covers 1 --max-steps 10
!python -m wmforge.report --run results/dryrun

## Full run

In [ ]:
!python -m wmforge.smoke --out {OUT} --stages clean real

In [ ]:
!python -m wmforge.smoke --out {OUT} --stages imprint

In [ ]:
from IPython.display import Image as show, Markdown, display
!python -m wmforge.report --run {OUT} > /dev/null
display(Markdown(open(f"{OUT}/summary.md").read()))
for name in ("trajectories.png", "grid.png"):
    if os.path.exists(f"{OUT}/{name}"):
        display(show(f"{OUT}/{name}"))

In [ ]:
!pip freeze > {OUT}/environment.txt
!nvidia-smi --query-gpu=name --format=csv,noheader > {OUT}/gpu.txt
!cd {OUT} && zip -q -r /content/checkin1_results.zip runs.csv summary.md trajectories.png grid.png environment.txt gpu.txt
from google.colab import files
files.download("/content/checkin1_results.zip")